In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
import dlt
from pyspark.sql.functions import col

@dlt.table(
    name="gold_layer.fact_sales",
    comment="Gold Layer: Central Sales Fact table capturing validated purchase transactions",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.zOrderCols": "event_time, product_id",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true",
        "delta.logRetentionDuration": "interval 30 days", # Required for Time Travel
        "delta.deletedFileRetentionDuration": "interval 7 days", # For VACUUM policy
        "delta.enableChangeDataFeed": "true" # Enables physical table features
    }
)
def fact_sales():
    """
    Fact table implementation: Filters Silver data for 'purchase' events 
    to drive revenue and conversion analytics
    """
    return (
        # 1. Read from the validated Silver layer
        dlt.read_stream("silver_layer.events_cleaned")
        
        # 2. Filter for transactions only (Business Logic)
        .filter(col("event_type") == "purchase")
        
        # 3. Select core metrics and foreign keys for the Star Schema
        .select(
            "user_id",       # FK to User Dimension (Optional)
            "product_id",    # FK to Product Dimension
            "event_time",    # Transaction timestamp
            "price",         # Quantitative metric for Revenue
            "user_session",   # Key for Customer Journey analysis
            "event_type"
        )
    )